exercise 1 Map Only
SELECT name, age
FROM users
WHERE age >= 18 AND country = 'FR'

In [ ]:
def map(key, row):

    if row.relation == "users" and row.age >= 18 and row.country == "FR":

        emit(null, (row.name, row.age))
        

Exercise 2
SELECT order_id
FROM orders
WHERE amount > 100

In [ ]:
def map(key, row):
    if row.relation == "orders" and row.amount > 100:
        emit(null, (row.order_id,))

Exercise 3
SELECT DISTINCT city
FROM customers

In [ ]:
def mapper(key, row):
    if row.relation == "customers":
        emit(row.city, 1)

def reducer(city, values):
        emit(city)

Exercise 4
SELECT user_id
FROM logins
UNION
SELECT user_id
FROM purchases

In [ ]:
def mapper(key, row):
    if row.relation == "logins":
        emit(row.user_id, 1)
    if row.relation == "purchases":
        emit(row.user_id, 1)


def reducer(user_id, values):
    emit(user_id)        

Exercise 5
SELECT product_id, COUNT(*)
FROM sales
GROUP BY product_id

In [ ]:
def mapper(key, row):
    if row.relation == "sales":
        emit(row.product_id, 1)

def reducer(product_id, values):
    count = sum(values)
    emit(product_id, count)




Exercise 6
SELECT country, AVG(salary)
FROM employees
GROUP BY country
HAVING COUNT(*) > 5

In [ ]:
def mapper(key, row):
    if row.relation == "employees":
        emit(row.country, (row.salary, 1))

def combiner(country, values):
    total_salary = sum(v.salary for v in values)      # if values are tuples
    total_count = sum(v.count for v in values)
    emit(country, (total_salary, total_count))

def reducer(country, values):
    total_salary = sum(v.salary for v in values)
    total_count = sum(v.count for v in values)
    if total_count > 5:
        avg_salary = total_salary / total_count
        emit(country, avg_salary)

Exercise 7
SELECT o.order_id, c.name
FROM orders o, customers c
WHERE o.customer_id = c.id

In [ ]:
def mapper(key, row):
    if row.relation == "o":
        emit(row.customer_id ,("o", row.order_id))
    if row.relation == "c":
        emit(row.customer_id, ("c",  row.name))


def reducer(customer_id, values):
    orders = [v[1] for v in values if v[0]== "o"]
    customers = [v[1] for v in values if v[0]=="c"]

    for order_id in orders:
        for name in customers:
            emit(order_id, name)

Exercise 8
SELECT c.country, SUM(o.amount)
FROM customers c, orders o
WHERE c.id = o.customer_id
GROUP BY c.country

In [ ]:
def mapper(key, row):
    if row.relation == "c":
        emit(row.id , ("c",row.country))
    if row.relation == "o":
        emit(row.customer_id, ("o",row.amount))


def reducer(customer_id, values):
    country = None
    total_amount = 0
    
    for source, data in values:
        if source == "c":
            country = data
        else:
            total_amount += data
    
    if country:
        emit(country, total_amount)

SELECT a, b, SUM(d)
FROM r1
WHERE c>5
GROUP BY a, b
HAVING COUNT(DISTINCT e)>1

In [ ]:
def mapper(key, row):
    if row.c > 5:
        emit((row.a, row.b), (row.d, row.e))

def combiner(key, values):
    sum_d = 0
    e_set = set()

    for d, e in values:
        sum_d += d
        e_set.add(e)

    emit(key, (sum_d, e_set))


def reducer(key, values):
    total_sum = 0
    global_e_set = set()

    for sum_d, e_set in values:
        total_sum += sum_d
        global_e_set.update(e_set)

    if len(global_e_set) > 1:
        emit(key[0], key[1], total_sum)


SELECT department,
       SUM(salary) AS total_salary,
       COUNT(DISTINCT employee_id) AS unique_employees
FROM employees
WHERE salary > 3000
GROUP BY department
HAVING COUNT(DISTINCT employee_id) > 2;


In [ ]:
def map(key, row):
    if row.relation == "employees" and row.salary>3000:
        emit(row.department, (row.salary, row.employee_id))


def combiner(key, values):
    sum_salary = 0
    employee_id_set = set()

    for salary, employee_id in values:
        sum_salary += salary
        employee_id_set.add(employee_id)
    
    emit(key, (sum_salary, employee_id_set))


def reducer(department, values):
    total_salary = 0
    all_ids = set()

    for sum_salary, id_set in values:
        total_salary += sum_salary
        all_ids.update(id_set)
    
    if len(all_ids)>2:
        emit(department, total_salary, len(all_ids))
